# 05 · ephys current steps: sag, input resistance, F–I, rebound

This notebook analyses the **current-step recordings** (A3 / M3): one 500 ms current step per sweep
(−150 to +100 pA in 50 pA steps in the example data), followed by 25 s at 0 pA in which rebound firing is seen.

Input: `02_ephys_anomaly-fix/output/analysis_ready.csv`. The injected current is the **command** waveform
(the current itself was not recorded: correction `command_as_current`), plus any amplifier bias current decided in 02.
Settings: `config.yaml` in this folder. Method: `README.md`.

| Step | What happens | Review |
|---|---|---|
| 1 | select the ready step recordings | which files |
| 2 | **detect spikes in every sweep and save a detection figure (PNG + SVG)** | open the figures: spikes, thresholds, step window |
| 3 | per-sweep and per-recording features, from the same spikes | Rin, first spiking step, F–I, sag, rebound |
| 4 | save | |

## Setup
`OVERWRITE = True` replaces existing output files. Close them in Excel first.

In [ ]:
OVERWRITE = True

from pathlib import Path

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

from ephys_analysis.spikes import SpikeSettings
from ephys_analysis.steps import StepSettings, detect_all, feature_table
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "05_ephys_steps")
sp = SpikeSettings.from_config(cfg["spikes"])
st = StepSettings.from_config(cfg["steps"])
out = REPO / "05_ephys_steps" / "output"
figures = out / "figures"
targets = [out / "spikes.csv", out / "attenuated_events.csv", out / "detection.csv",
           out / "step_sweeps.csv", out / "step_features.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)
print(st)

## Step 1 · Select the current-step recordings

In [ ]:
ready = pd.read_csv(repo_path(cfg, cfg["recordings_table"]), dtype={"cell_tag": "string"})
selection = ready[ready["family"].isin(cfg["families"])].reset_index(drop=True)
display(selection[["recording_id", "cell_tag", "abf_filename", "voltage_source", "current_source", "bias_current_pA", "corrections"]])

## Step 2 · Detect spikes and save a detection figure for every file
The step window is found from the command waveform (the sweep with the largest step); each sweep's step
amplitude is the command during the step minus the command just before it. Spikes are detected in the
**whole** sweep with the same detector as 04 and labelled by period: **pre** (before the step, spontaneous),
**step**, **post** (after the step: rebound).

Figure per recording (`output/figures/<recording_id>.png` and `.svg`):
- top: the step period of every sweep (colour = current), spikes marked (grey = flagged);
- right: whole sweeps, stacked, so rebound firing in the 25 s after the step is visible;
- bottom: every spike aligned to its threshold (black dot).

Spikes during strong, sustained firing often have a slow upstroke (< 50 mV/ms) or a half-width > 5 ms: they are
**counted** (firing rate) but **flagged**, and their waveform is not used.

**Attenuated events** (open circles ○): voltage peaks ≥ 20 mV above their surroundings that do *not* pass the
spike criteria (dV/dt never reaches 20 mV/ms). During strong steps some cells go into **depolarisation block**:
after one or two full spikes the peaks shrink and rise slowly. These events are **not counted as spikes**; they are
listed in `attenuated_events.csv`, counted per sweep (`n_attenuated_step`) and the recording is flagged
`depolarisation_block`.

In [ ]:
spikes, events, detection = detect_all(selection, sp, st, log, figures, tuple(cfg["figure_formats"]))
display(detection)

In [ ]:
# Review: every detection figure. If detection is wrong, change `spikes:` in config.yaml and re-run Step 2.
for f in detection["figure"].dropna():
    if f:
        print(Path(f).name)
        display(Image(filename=f, width=1100))

**Checkpoint.** Only continue when the detection is right: the features below are computed from exactly
these spikes.

## Step 3 · Features
**Per sweep** (`step_sweeps.csv`): Vm before the step, spikes during the step (rate, latency, ISI CV, adaptation),
v_min / v_ss and **sag ratio** = (v_ss − v_min) / (vm_pre − v_min) for hyperpolarising steps, and **rebound**:
peak depolarisation in the first 500 ms, spikes in the first 2 s and to the end of the sweep, bursts.

**Per recording** (`step_features.csv`):
- **Rin** = slope of (v_ss − vm_pre) against the step current over the hyperpolarising steps;
- **first spiking step** = smallest positive step with ≥ 1 spike (coarse: 50 pA; the rheobase comes from the 06 ramp);
- **F–I slope** = linear fit of step firing rate from the first spiking step upwards;
- sag and rebound at the most negative step; **AP waveform** = first valid spike at the first spiking step.

In [ ]:
sweeps, features = feature_table(selection, spikes, events, st, log)
display(sweeps[["abf_filename", "sweep", "I_step_pA", "I_total_pA", "vm_pre_mV", "n_spikes_pre", "n_spikes_step",
                "rate_step_Hz", "latency_first_spike_ms", "v_min_mV", "v_ss_mV", "sag_ratio", "rebound_peak_mV",
                "n_rebound_spikes_short", "n_rebound_spikes_all", "n_attenuated_step"]].round(2))

In [ ]:
display(features.T)

## Step 4 · Save

In [ ]:
for df, p in zip((spikes, events, detection, sweeps, features), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)